# Stockmog phase 2: buy / sell / stay
Upload `ppo-dataset-top4.tar` AND `ppo-dataset-top4.sha256.json` from the repository's `google_colab` folder to `MyDrive/Stockmog` (or the matching full-universe pair when `USE_TOP_PROFIT_ARCHIVE = False`). This notebook does not upload data or place live trades. Only open archives you trust: they include Python source and model weights.

One independent MaskablePPO policy per stock/timeframe. Actor and critic each use **512 -> 512 -> 256** hidden units. News is low priority: a separate news branch can change each action logit by at most **+/-0.05**. Primary actor/critic inputs exclude news and its missing-data masks; the critic never uses news. News can influence near-ties, not overturn a primary action-score gap greater than 0.10. Buy opens a long position, sell closes it, stay waits when flat or holds shares. No shorting, pyramiding, or forecast-price fills. Whole shares only. Defaults: Rs 50,000, 10% exposure, 0.10% fees and 0.05% slippage per side.

**News is sampled/incomplete**, not a complete historical archive. GKG historical releases and recent DOC metadata are separate endpoints of the same free provider. Check coverage below. No publisher article bodies are included. News does not alter recorded market prices. Later market reactions are outcomes, not advance inputs.

The saved price model is frozen before PPO's data begins. Its former supervised holdout becomes new chronological RL train/validation/test data (60/20/20 by session). Previously viewed report dates are not a pristine prospective test. Sparse daily/weekly histories can severely overfit a large network. This is a research pipeline, not evidence of profitable trading.

**Trend-analyser stream:** each stock/timeframe's frozen XGBoost + LSTM trend analyser runs over the history and feeds its predicted next-candle log return (`model_log_return`) to PPO, alongside cross-timeframe returns/RSI and external factors. By default this notebook uses `ppo-dataset-top4.tar`, which only contains the trend analyser's positive-profit picks (M&M, MAXHEALTH, ONGC, BSE): 36 jobs instead of 543.

**Research/report stream (priority 5/10, news 1-2/10)** for the profitable picks M&M, MAXHEALTH, ONGC and BSE: NSE filings (results, analyst meets, presentations, credit ratings, corporate/legal actions), quarterly profit/revenue YoY from NSE XBRL, GDELT research-style headlines and optional user broker reports (rating, target price). Each item is used only after exchange/publication time + 15 minutes. A separate research branch can shift each action logit by at most **+/-0.15**; like news it is excluded from the primary actor and critic, starts at zero weight and is missing (masked) for other stocks. Paywalled broker research is not included unless you supply it.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
TOP_PROFIT_SYMBOLS = ['M&M', 'MAXHEALTH', 'ONGC', 'BSE']  # Positive-profit trend-analyser picks.
USE_TOP_PROFIT_ARCHIVE = True  # False uses the full 57-stock ppo-dataset.tar.
ARCHIVE = Path('/content/drive/MyDrive/Stockmog/' + ('ppo-dataset-top4.tar' if USE_TOP_PROFIT_ARCHIVE else 'ppo-dataset.tar'))
OUTPUT = Path('/content/drive/MyDrive/Stockmog/ppo-policies-research')
WORK = Path('/content/stockmog-training-research')
ALLOW_PARTIAL_NEWS = False  # Set True only after reviewing the coverage report.
ALLOW_PARTIAL_RESEARCH = False  # Set True after reviewing research-coverage.json.
RUN_FULL_TRAINING = False
RUN_FINAL_TEST = False
STEPS_PER_JOB = 262_144
NEWS_LOGIT_LIMIT = 0.05  # Set 0 to disable news's direct effect on action scores.
RESEARCH_LOGIT_LIMIT = 0.15  # Research priority 5/10; set 0 to disable, max 0.3.
SYMBOLS = TOP_PROFIT_SYMBOLS if USE_TOP_PROFIT_ARCHIVE else None  # None selects every exported stock.
TIMEFRAMES = None  # None selects all nine timeframes.


In [ ]:
import hashlib, json, shutil, tarfile
def file_hash(path):
    h = hashlib.sha256()
    with path.open('rb') as f:
        for chunk in iter(lambda: f.read(8 * 1024**2), b''):
            h.update(chunk)
    return h.hexdigest()
sidecar = json.loads(ARCHIVE.with_suffix('.sha256.json').read_text())
assert ARCHIVE.stat().st_size == sidecar['bytes'], 'Archive size mismatch'
assert file_hash(ARCHIVE) == sidecar['sha256'], 'Archive checksum mismatch'
WORK.mkdir(parents=True, exist_ok=True)
BUNDLE = WORK / 'stockmog-ppo'
extraction_marker = WORK / 'archive-sha256.txt'
if (BUNDLE / 'manifest.json').exists():
    if not extraction_marker.exists() or extraction_marker.read_text().strip() != sidecar['sha256']:
        raise RuntimeError('Archive changed or old extraction is unverified. Choose a fresh WORK directory.')
else:
    assert shutil.disk_usage(WORK).free > ARCHIVE.stat().st_size * 1.15, 'Insufficient local disk'
    with tarfile.open(ARCHIVE) as archive:
        archive.extractall(WORK, filter='data')
    extraction_marker.write_text(sidecar['sha256'])
manifest = json.loads((BUNDLE / 'manifest.json').read_text())
assert manifest['status'] == 'complete'
print('Jobs:', len(manifest['jobs']))
print('RL-eligible jobs:', manifest['rl_eligible_jobs'])
print('Insufficient history:', [(j['symbol'], j['timeframe'], j['rl_ineligible_reason']) for j in manifest['jobs'] if not j['rl_eligible']])
print(json.dumps(json.loads((BUNDLE / 'news-coverage.json').read_text()), indent=2))
if (BUNDLE / 'research-coverage.json').exists():
    print(json.dumps(json.loads((BUNDLE / 'research-coverage.json').read_text()), indent=2))
else:
    print('No research stream in this archive.')
print('\n'.join(manifest['notes']))


In [ ]:
import os, subprocess, sys
os.environ.setdefault('OMP_NUM_THREADS', '2')
SOURCE = BUNDLE / 'source'
for name, expected in manifest['files'].items():
    if name.startswith('source/'):
        assert file_hash(BUNDLE / name) == expected, f'Source checksum mismatch: {name}'
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(SOURCE / 'requirements-rl.txt')])
sys.path.insert(0, str(SOURCE))
import torch
from data_pipeline.rl_colab_training import PPOConfig, train_jobs
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none')
print('Torch:', torch.__version__)


## A100 benchmark: can this runtime reach 500 steps/sec?
Select A100 under **Runtime > Change runtime type**, run the setup cells, and review the news and research coverage before setting `ALLOW_PARTIAL_NEWS = True` and `ALLOW_PARTIAL_RESEARCH = True`. Keep `RUN_FULL_TRAINING = False` during the pilot. The benchmark trains fresh 1m and 1d pilots on local disk, prints actual steps/sec, and deletes the temporary policies. Enter the compute units/hour shown by Colab in `CU_PER_HOUR` if known. To compare with CPU or another GPU, use the same pilot settings. Never use final-test profit to select hardware or settings.

The original multi-year history is included, but PPO cannot use the saved predictor before its freeze date. A genuinely longer model-informed RL history requires new walk-forward supervised training, not back-predicting with a model that saw future labels.

In [ ]:
from tempfile import TemporaryDirectory
from time import perf_counter
from math import ceil, isfinite

DEVICE = 'cuda'  # Set 'cpu' to run a CPU comparison.
CU_PER_HOUR = None  # Enter your runtime's displayed rate, e.g. 5.5.
CU_BUDGET = 600
TARGET_STEPS_PER_SECOND = 500
BENCHMARK_STEPS = 8192  # Increase to 32768 for a less noisy follow-up.
BENCHMARK_SYMBOL = 'M&M'  # Exercises the research stream.
BENCHMARK_TIMEFRAMES = ['1m', '1d']

if DEVICE == 'cuda' and not torch.cuda.is_available():
    raise RuntimeError('No CUDA GPU found. Select A100 in Colab or set DEVICE to cpu.')
assert manifest['news_complete'] or ALLOW_PARTIAL_NEWS, 'Review news coverage, then explicitly allow partial news.'
assert ('research.parquet' not in manifest['files'] or manifest.get('research_complete')
        or ALLOW_PARTIAL_RESEARCH), 'Review research coverage, then explicitly allow partial research.'
if CU_PER_HOUR is not None and (not isfinite(CU_PER_HOUR) or CU_PER_HOUR <= 0):
    raise ValueError('CU_PER_HOUR must be positive, or None if unknown.')
selected_jobs = [j for j in manifest['jobs'] if (not SYMBOLS or j['symbol'] in SYMBOLS)
                 and (not TIMEFRAMES or j['timeframe'] in TIMEFRAMES) and j['rl_eligible']]
if not selected_jobs:
    raise ValueError('No eligible full-run jobs match SYMBOLS and TIMEFRAMES.')
config = PPOConfig(steps=BENCHMARK_STEPS, device=DEVICE, news_logit_limit=NEWS_LOGIT_LIMIT,
                   research_logit_limit=RESEARCH_LOGIT_LIMIT)
print('Benchmark hardware:', torch.cuda.get_device_name(0) if DEVICE == 'cuda' else 'CPU')
with TemporaryDirectory(prefix='ppo-benchmark-', dir=WORK) as scratch:
    started = perf_counter()
    benchmark = train_jobs(BUNDLE, Path(scratch), config, symbols=[BENCHMARK_SYMBOL],
                           timeframes=BENCHMARK_TIMEFRAMES, allow_partial_news=ALLOW_PARTIAL_NEWS,
                           allow_partial_research=ALLOW_PARTIAL_RESEARCH)
    wall_seconds = perf_counter() - started
jobs = benchmark['jobs']
if benchmark['status'] != 'complete' or {j['timeframe'] for j in jobs} != set(BENCHMARK_TIMEFRAMES):
    raise RuntimeError('Benchmark did not complete both requested pilots.')
for job in jobs:
    seconds = job['elapsed_seconds']
    print(f"{job['symbol']} {job['timeframe']}: {job['timesteps']:,} steps in {seconds:.1f}s ({job['timesteps'] / seconds:.1f} steps/sec)")
pilot_steps = sum(j['timesteps'] for j in jobs)
speed = pilot_steps / wall_seconds
target_status = 'met' if speed >= TARGET_STEPS_PER_SECOND else 'not met'
print(f'Combined: {pilot_steps:,} steps in {wall_seconds:.1f}s = {speed:.1f} steps/sec. Target {TARGET_STEPS_PER_SECOND}: {target_status}.')
rollout_size = config.rollout * config.envs
full_steps = len(selected_jobs) * ceil(STEPS_PER_JOB / rollout_size) * rollout_size
rough_hours = full_steps / speed / 3600
print(f'Projected {len(selected_jobs)} jobs / {full_steps:,} steps: {rough_hours:.1f} active hours ({rough_hours / 24:.1f} days).')
print('Pilot includes prep, inference and validation. Projection scales those fixed costs and excludes initial setup, Drive checkpoint I/O and interruptions.')
if CU_PER_HOUR is not None:
    required_cu = rough_hours * CU_PER_HOUR
    print(f'Projected {required_cu:.1f} CU versus {CU_BUDGET:g} CU budget (before overhead).')
else:
    print('Set CU_PER_HOUR to estimate compute use; Colab rates vary.')


In [ ]:
config = PPOConfig(steps=STEPS_PER_JOB, device=DEVICE, news_logit_limit=NEWS_LOGIT_LIMIT,
                   research_logit_limit=RESEARCH_LOGIT_LIMIT)
if RUN_FULL_TRAINING:
    training = train_jobs(BUNDLE, OUTPUT, config, symbols=SYMBOLS, timeframes=TIMEFRAMES,
                          allow_partial_news=ALLOW_PARTIAL_NEWS,
                          allow_partial_research=ALLOW_PARTIAL_RESEARCH)
    print('Completed jobs:', len(training['jobs']))
else:
    print('Full training disabled. Review benchmark, coverage and budget first.')
print('Rerun with unchanged archive/settings to resume verified per-job checkpoints.')


In [ ]:
if RUN_FINAL_TEST:
    final = train_jobs(BUNDLE, OUTPUT, config, symbols=SYMBOLS, timeframes=TIMEFRAMES,
                       allow_partial_news=ALLOW_PARTIAL_NEWS,
                       allow_partial_research=ALLOW_PARTIAL_RESEARCH, evaluate_test=True)
    import pandas as pd
    results = pd.DataFrame([{'stock': j['symbol'], 'timeframe': j['timeframe'],
                             'profit_rs': j['ppo']['profit'], 'return_pct': j['ppo']['return_percent'],
                             'max_drawdown_pct': j['ppo']['max_drawdown_percent'],
                             'buy_hold_profit_rs': j['buy_hold']['profit'],
                             'open_shares': j['ppo']['open_shares']} for j in final['jobs']])
    results.to_csv(OUTPUT / 'test-results.csv', index=False)
    display(results.sort_values('profit_rs', ascending=False))
else:
    print('Final test disabled. Enable only after fixing the training design.')
